# ForestGeo Risk Analyser — live demo

**Where do wildfire, roads, settlements and wildlife overlap in The Nilgiris, and which villages are most exposed?**

Four unrelated open datasets, one spatial database, one answer.

| theme | source | contributes |
|---|---|---|
| fire | NASA FIRMS (VIIRS satellite) | 1,564 thermal detections, Jan–May 2024 |
| settlement & access | OpenStreetMap | 421 villages, 37,665 roads/tracks/railways |
| habitat | OpenStreetMap | 119 forest areas, 9 protected areas |
| wildlife | GBIF | 2,661 mammal sightings, 72 species |

This notebook is a **thin client**. All the logic lives in the `forestgeo` package on GitHub; the data lives in **MongoDB Atlas**. Nothing is recomputed here — the queries run against the cloud database in real time.


## 1. Setup

Installs the client libraries and clones the project package. Takes about a minute.

Note what is **not** installed: `geopandas` and `osmnx` are only needed for fetching and cleaning, which already happened. The demo only needs to query and draw.


In [ ]:
!pip -q install pymongo shapely pyproj folium python-dotenv certifi 2>/dev/null
!git clone -q https://github.com/NithiishSD/bigdata_spatial_project.git 2>/dev/null || echo "already cloned"
%cd /content/bigdata_spatial_project
!pip -q install -e . 2>/dev/null
print("ready")

## 2. Connect to MongoDB Atlas

The connection string contains a password, so it is **never written in this notebook**. It is read from Colab's Secrets manager.

**To set it up:** click the 🔑 key icon in the left sidebar → *Add new secret* → name it `MONGODB_URI`, paste your Atlas connection string as the value, and enable *Notebook access*.


In [ ]:
import os
from google.colab import userdata

os.environ["MONGODB_URI"] = userdata.get("MONGODB_URI")
os.environ["MONGODB_DB"] = "forestgeo"

from forestgeo.db import get_db, safe_uri

db = get_db()
print("connected to:", safe_uri())        # password redacted
print("ping:", db.command("ping"))
print("server:", db.client.server_info()["version"])

## 3. What is in the database

Ten collections, every geometry validated in Shapely before insert, every collection carrying a **2dsphere** index.


In [ ]:
import pandas as pd
from forestgeo.db import verify

rep = verify(db)
display(rep[["collection", "count", "geom_types", "has_2dsphere", "n_indexes"]])
print(f"total documents: {int(rep['count'].sum()):,}")

All three GeoJSON vector types are present and indexed:

* **Point** — villages, fire hotspots, wildlife sightings
* **LineString / MultiLineString** — roads, tracks, railways, rivers
* **Polygon / MultiPolygon** — forests, protected areas, water, farmland, and the computed zones

46,779 raw features were reduced to 46,208 valid ones. **Zero were rejected by MongoDB** — the validation stage caught everything first.


## 4. The four spatial operations

| operator | question | needs an index? | countable? |
|---|---|---|---|
| `$nearSphere` | what is closest, in order? | **yes** | no |
| `$geoNear` | the same, but returns the distance | **yes** | n/a |
| `$geoWithin` | what is entirely inside? | no | yes |
| `$geoIntersects` | what touches or crosses? | no | yes |


In [ ]:
# $geoWithin — what lies inside Mudumalai National Park
pa = db.protected_areas.find_one({"name": {"$regex": "Mudumalai"}})
inside = {"geometry": {"$geoWithin": {"$geometry": pa["geometry"]}}}

print(f"{pa['name']}  ({pa['area_km2']:.0f} km2)")
print("  sightings inside :", db.sightings.count_documents(inside))
print("  hotspots inside  :", db.fire_hotspots.count_documents(inside))

# $geoIntersects — roads CROSSING it.
# A road that enters and leaves is not *entirely inside*, so $geoWithin would miss it.
crosses = {"geometry": {"$geoIntersects": {"$geometry": pa["geometry"]}}}
print("  roads crossing   :", db.transport.count_documents({**crosses, "kind": "road"}))
print("  roads *within*   :", db.transport.count_documents({**inside, "kind": "road"}),
      "  <- why $geoIntersects is the right operator for lines")

In [ ]:
# $nearSphere — nearest first, sorted by distance. $maxDistance is in METRES.
centre = {"type": "Point", "coordinates": [76.65, 11.40]}   # [longitude, latitude]

for v in db.villages.find(
        {"geometry": {"$nearSphere": {"$geometry": centre, "$maxDistance": 8000}}},
        {"name": 1, "_id": 0}).limit(5):
    print(" ", v.get("name"))

In [ ]:
# $geoWithin + $centerSphere — counting within a radius.
# $nearSphere cannot be used in count_documents(), and $centerSphere's radius is in
# RADIANS, not metres: kilometres / 6378.1 (the Earth's radius).
from forestgeo.config import EARTH_RADIUS_KM

for km in (2, 5, 10):
    n = db.fire_hotspots.count_documents(
        {"geometry": {"$geoWithin": {"$centerSphere": [[76.65, 11.40], km / EARTH_RADIUS_KM]}}})
    print(f"  hotspots within {km:>2} km of the region centre: {n}")

### Protected areas, ranked

The clearest finding in the project: **fire pressure and wildlife recording are almost inverse.**


In [ ]:
from forestgeo.queries import protected_area_counts
display(protected_area_counts(db))

Sathyamangalam Tiger Reserve records the **most fire detections (52)** and the **fewest wildlife sightings (14)**. Mudumalai is the opposite — 533 sightings at 1.589 per km².


## 5. The union — what MongoDB cannot do

MongoDB can *test* spatial relationships, but it cannot **construct** geometry. There is no union operator, no intersection, no buffer that returns a new shape.

So the construction happens in **Shapely**, in a metre-based projection, and the result is written back into a `derived_zones` collection — where ordinary MongoDB operators can query it.

```
MongoDB points  →  Shapely (buffer, union, intersect)  →  MongoDB polygons  →  $geoWithin
```


In [ ]:
zones = pd.DataFrame(list(db.derived_zones.find(
    {}, {"_id": 0, "zone_type": 1, "input_count": 1, "n_parts": 1,
         "area_km2": 1, "params": 1})))
display(zones)

In [ ]:
# The computed geometry is now just another collection to query.
burn = db.derived_zones.find_one({"zone_type": "burn_risk_footprint"})
crit = db.derived_zones.find_one({"zone_type": "critical_zone"})

in_burn = {"geometry": {"$geoWithin": {"$geometry": burn["geometry"]}}}
in_crit = {"geometry": {"$geoWithin": {"$geometry": crit["geometry"]}}}

print("villages inside the burn-risk footprint:", db.villages.count_documents(in_burn))
print("villages inside the critical zone      :", db.villages.count_documents(in_crit))
print("hotspots inside the critical zone      :", db.fire_hotspots.count_documents(in_crit))

### Validating Shapely against MongoDB

The critical zone was built in Shapely as *burn footprint ∩ habitat buffer*. A village inside it must therefore also be inside **both** source zones — a question MongoDB can answer directly with two predicates combined by `$and`.

If the two counts agree, the Shapely intersection is confirmed by the database's own spatial reasoning. Two engines, two representations, one answer.


In [ ]:
from forestgeo.cross_theme import two_predicate_equivalence
for k, v in two_predicate_equivalence(db).items():
    print(f"  {k:28} {v}")

## 6. The cross-theme answer

Every theme combined into one ranked list: hotspot density, distance to the nearest fire, and membership of the computed zones.


In [ ]:
from forestgeo.cross_theme import village_risk, features_in_critical_zone

display(features_in_critical_zone(db))

risk = village_risk(db)
print(f"\n{len(risk)} villages scored | "
      f"{int(risk.in_critical_zone.sum())} in the critical zone | "
      f"{int(risk.in_burn_footprint.sum())} in the burn footprint")
display(risk[["village", "n_hotspots_5km", "nearest_hotspot_m",
              "n_sightings_5km", "in_critical_zone", "risk_score"]].head(10))

## 7. Does the index matter?

Measured with `explain("executionStats")`, which reports time **inside the server** rather than wall-clock time, so network latency to Atlas is excluded.

`.hint({"$natural": 1})` forces a collection scan without dropping the index.


In [ ]:
from forestgeo.benchmark import timed
from forestgeo.config import EARTH_RADIUS_KM

filt = {"geometry": {"$geoWithin": {"$centerSphere": [[76.70, 11.41], 5 / EARTH_RADIUS_KM]}}}

rows = []
for natural in (False, True):
    r = timed(db, "sightings", filt, natural, runs=3)
    rows.append({"mode": "COLLSCAN" if natural else "indexed",
                 "stage": r["stage"], "returned": r["n_returned"],
                 "docs_examined": r["docs_examined"],
                 "ms": r["millis_median"]})
display(pd.DataFrame(rows))

The indexed query examines only the documents it needs; the scan reads the whole collection to return the same rows.

**The scale test, run locally on 250,000 synthetic points:**

| selectivity | indexed | COLLSCAN | gain |
|---|---|---|---|
| **~0.2 %** | **1 ms**, 748 docs examined | 180 ms, 250,000 examined | **180× / 334×** |
| ~25 % | 154 ms, 84,972 examined | 197 ms, 250,000 examined | 1.3× |

The second row matters as much as the first: **an index accelerates *selective* queries.** At 25 % selectivity the index must read a quarter of the entries *and then fetch* a quarter of the documents, while the scan reads everything once with no lookups. Past roughly a third selectivity, scanning wins — which is why query planners sometimes ignore an index deliberately.


In [ ]:
from IPython.display import Image, display as show
show(Image("outputs/benchmark.png", width=950))

### The operators that *require* an index

`$geoWithin` and `$geoIntersects` fall back to a collection scan. `$nearSphere` does not — it refuses to run at all.


In [ ]:
from pymongo.errors import OperationFailure

probe = db["_demo_probe"]
probe.drop()
probe.insert_one({"geometry": {"type": "Point", "coordinates": [76.65, 11.40]}})

try:
    list(probe.find({"geometry": {"$nearSphere": {"$geometry": centre}}}).limit(1))
    print("unexpectedly succeeded")
except OperationFailure as exc:
    print("$nearSphere without an index ->", exc.details.get("errmsg", "")[-60:])

n = probe.count_documents(
    {"geometry": {"$geoWithin": {"$centerSphere": [[76.65, 11.40], 5 / EARTH_RADIUS_KM]}}})
print(f"$geoWithin  without an index -> works, returned {n} (by scanning)")
probe.drop()

## 8. The map

Built live from Atlas: layered themes, villages coloured by risk, and the fire season animated day by day.


In [ ]:
from forestgeo.mapviz import build
path = build(db, risk)
print("built:", path)

In [ ]:
from IPython.display import IFrame
# Rendered in an iframe: the map is ~10 MB and inlining it would slow the notebook.
IFrame(src="outputs/forestgeo_map.html", width="100%", height=620)

## 9. What this demonstrates

1. **All three GeoJSON vector types**, stored and 2dsphere-indexed in one database.
2. **Validation before storage** — 46,779 raw features → 46,208 valid, every discard counted by reason, **zero rejected** by MongoDB.
3. **All four spatial operators**, each used where it is correct — `$geoIntersects` for lines crossing areas, `$geoWithin` + `$centerSphere` for counting in a radius, `$geoNear` where the distance itself is needed.
4. **The database's limits, handled honestly** — MongoDB cannot build geometry, so the union happens in Shapely and returns as a queryable collection, then is validated against MongoDB's own reasoning.
5. **Measured index performance** — 180× on a selective query, 1.3× on a broad one. Both numbers matter.

### Limitations worth stating

* **88.7 % of sightings lie within 500 m of a road.** That is *observer bias*, not animal behaviour — people record wildlife where people are. A roadkill dataset contributing 33 % of precise records was excluded for the same reason; including it raises the figure to 92.4 %.
* **24.7 % of wildlife records are deliberately obscured** to about 31 km, because they are tiger and elephant locations and the displacement prevents poaching. It is irreversible — 659 records, 659 distinct coordinates, no grid to invert — so records are *tiered by precision* rather than filtered, and distance queries use only the precise tier.
* **A hotspot is a thermal anomaly, not a confirmed fire.** Industrial heat causes false positives; fire under cloud or between overpasses is missed entirely. Hence the 1 km buffer rather than treating detections as exact points.
* **OSM is a current snapshot**, so 2026 roads are compared against 2024 fires. Overpass supports date-filtered attic queries, which would fix this at the cost of considerable complexity.
